# Parquet files on MinIO

List keys, then read one Parquet file with PyArrow (path-style S3 to MinIO).

In [4]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password",
    region_name="us-east-1",
)

prefix = "default/public/demo_orders/"
resp = s3.list_objects_v2(Bucket="warehouse", Prefix=prefix, MaxKeys=20)
keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".parquet")]
keys

['default/public/demo_orders/1775841544257.parquet',
 'default/public/demo_orders/1775841547948.parquet',
 'default/public/demo_orders/1775841551540.parquet',
 'default/public/demo_orders/1775841555627.parquet']

In [9]:
import pyarrow.dataset as ds
from pyarrow import fs

if not keys:
    raise SystemExit("No parquet keys yet — run scripts/load.sh")

path = f"warehouse/{keys[0]}"
dataset = ds.dataset(
    path,
    format="parquet",
    filesystem=fs.S3FileSystem(
        endpoint_override="http://127.0.0.1:9000",
        access_key="admin",
        secret_key="password",
        scheme="http",
    ),
)
dataset.to_table()

pyarrow.Table
__operation: string not null
__idempotency_key: int64 not null
__event_time: int64 not null
id: int64 not null
customer_id: int64 not null
amount: decimal128(12, 2) not null
created_at: timestamp[us, tz=UTC] not null
----
__operation: [["INSERT","INSERT","INSERT","INSERT","INSERT",...,"INSERT","INSERT","INSERT","INSERT","INSERT"]]
__idempotency_key: [[30802296,30802544,30802696,30802896,30803048,...,30803552,30803704,30803904,30804056,30804208]]
__event_time: [[1775841543137,1775841543137,1775841543137,1775841544147,1775841544147,...,1775841545159,1775841545159,1775841546175,1775841546175,1775841546175]]
id: [[1,2,3,4,5,...,8,9,10,11,12]]
customer_id: [[9715,9080,1091,4043,9722,...,1684,2997,4210,9425,2551]]
amount: [[203.28,180.57,62.30,427.89,358.09,...,283.00,384.30,185.04,302.39,465.87]]
created_at: [[2026-04-10 17:19:03.135012Z,2026-04-10 17:19:03.135012Z,2026-04-10 17:19:03.135012Z,2026-04-10 17:19:04.144091Z,2026-04-10 17:19:04.144091Z,...,2026-04-10 17:19:05.15519